*0.3 Classical NLP*

# Stemming

**The situation.** A keyword search over 4,000 help articles. A customer searches "refunds". The article is titled "How to request a refund". No match. "charging" does not find "charged"; "invoices" does not find "invoice". Every plural and verb form is a missed result.

**Stemming.** Chop word endings by rule so related forms collapse to one stem: refunds → refund, charging → charg, invoices → invoic. The stems are not always real words — they do not need to be; they only need to be the *same* for the forms you want to match. Fast, no dictionary, and built into search engines (Elasticsearch, Lucene).

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")
words = [
    "refund",
    "refunds",
    "refunded",
    "refunding",
    "charge",
    "charged",
    "charging",
    "invoice",
    "invoices",
    "universal",
    "university",
]
print(f"{'word':<12}{'porter':>10}{'snowball':>10}")
for word in words:
    print(f"{word:<12}{porter.stem(word):>10}{snowball.stem(word):>10}")
assert porter.stem("refunds") == porter.stem("refunded") == "refund"

word            porter  snowball
refund          refund    refund
refunds         refund    refund
refunded        refund    refund
refunding       refund    refund
charge           charg     charg
charged          charg     charg
charging         charg     charg
invoice         invoic    invoic
invoices        invoic    invoic
universal      univers   univers
university     univers   univers


**Reading the output.** All the refund forms collapse to `refund`, the charge forms to `charg`, the invoice forms to `invoic`. And the last two lines show the cost: "universal" and "university" collapse to the same stem though they mean different things.

**Search, before and after.** Stem both the query and the titles; the plural now matches.

In [3]:
titles = [
    "How to request a refund",
    "Why was I charged twice?",
    "Download your invoices",
    "Change your password",
]


def stems(text: str) -> set[str]:
    result = set()
    for word in text.lower().split():
        result.add(snowball.stem(word.strip("?.,")))
    return result


query = "refunds charging"
exact_hits = []
stemmed_hits = []
for title in titles:
    if set(query.split()) & set(title.lower().split()):
        exact_hits.append(title)
print("exact-word matches:", exact_hits)
for title in titles:
    if stems(query) & stems(title):
        stemmed_hits.append(title)
print("stemmed matches:   ", stemmed_hits)
assert len(stemmed_hits) == 2

exact-word matches: []
stemmed matches:    ['How to request a refund', 'Why was I charged twice?']


**The rule to remember.** Stem for keyword search and counting, where "same stem" is all you need. It is a crude, fast rule; accept that some unrelated words will collide.

| Use it when | Don't when | Instead use |
|---|---|---|
| keyword search, TF-IDF/BM25 features, word counts | the output is shown to people, or meaning must be preserved | lemmatization (next item) |

**Watch out**
- Never stem text that goes into an LLM or an embedding model; they handle word forms themselves and stems confuse them.
- Snowball is the modern Porter; use it. Both are English-only unless you pick the language.
- Over-stemming ("university" = "universal") is the known failure; check your top queries.